# Кандидатогенерация для поиска услуг Авито — полное решение (Recall@50 = 0.9307)

Ноутбук целиком воспроизводит итоговое решение: подготовку данных, локальную валидацию, обучение энкодеров и ранкера, сборку `answer.csv`. Подробное описание подхода, анализа данных и ошибок — в `README.md`.

**Итоговая конфигурация** (выбрана этим же ноутбуком по валидации):
- **основной dense-энкодер:** `Qwen/Qwen3-Embedding-0.6B`, дообученный в два раунда — in-batch негативы, затем hard negatives из той же локации;
- **второй dense-признак:** дообученный `intfloat/multilingual-e5-small`;
- **ранжирование:** кандидаты (BM25, dense, символьные n-граммы, микрокатегории, kNN по логу) → ансамбль LightGBM LambdaRank → правила категорий.

**Разделы ноутбука:**

| Раздел | Что делает |
|---|---|
| 0 | подготовка данных и псевдо-бенчмарк для валидации (`scripts/prep.py … split.py`) |
| 1 | метрики: `R@50` и прокси лидерборда `R@35w` |
| 2 | bi-encoder: обёртка, дообучение (InfoNCE), кодирование; 2.1 — проверка готовых моделей, если они есть |
| 3 | тест энкодеров на масштаб (dense-only): e5-small / base / large, USER-bge-m3, Qwen3-Embedding-0.6B |
| 4 | майнинг hard negatives и второй раунд дообучения лучшего энкодера |
| 5 | полный пайплайн: кандидаты и признаки (`src/pipeline.py`) + LightGBM; три варианта энкодеров |
| 6 | cross-encoder вторым проходом ранкера (проверен и **не** вошёл в финал: +0.02 п.п.) |
| 7 | сводка валидации |
| 8 | финал: энкодеры переобучаются на всём train → `subs/sub_server_stage1.csv` = отправленный `answer.csv` |

**Метрики.** `R@50` и `R@35w` — полнота@35 на валидации, перевзвешенной под состав бенчмарка. Корпус бенчмарка примерно в 1.45 раза плотнее валидационного, поэтому 50 слотов там работают примерно как 35 на валидации. Проверено на трёх отправках: лидерборд ≈ `R@35w` − ~1 п.п.

**Без утечек.** Всё, что даёт признаки ранкеру на валидации (энкодеры, cross-encoder, приоры), обучается только на T-логе. Запросы rk/val и их объявления в обучение не попадают. Для бенчмарка те же рецепты переобучаются на всём train.

**Запуск.** Положите `train.parquet`, `benchmark_queries.parquet`, `benchmark_items.parquet` в корень репозитория (рядом с `src/`) и выполните все ячейки. Шаги кешируются в `models/` и `data/`, прерванный запуск продолжается с места остановки. Время на одной GPU уровня A100 / RTX PRO 6000 — около 6–8 часов. Для быстрой проверки кода: `SMOKE = True` (крошечные подвыборки, минуты).

**Воспроизводимость.** Сиды зафиксированы (torch / numpy / LightGBM), разбиение данных детерминировано. Обучение нейросетей на GPU в bf16 не бит-в-бит детерминировано, поэтому повторный запуск даёт тот же результат с точностью до мелких перестановок у 50-й позиции.

In [ ]:
# ==== Конфигурация ====
import os, sys, time, json, math, random, gc, re, pickle, subprocess
from pathlib import Path
import numpy as np, pandas as pd

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')   # fork при сборке признаков
os.environ.setdefault('HF_XET_CHUNK_CACHE_SIZE_BYTES', '0')   # не держать кеш чанков загрузок (до 10 ГБ на диске)

PROJECT_DIR = None       # можно указать явно, например '/workspace/cand-gen'; иначе ищется автоматически

def find_root():
    """Корень проекта = папка с train.parquet и benchmark_items.parquet. Порядок поиска: PROJECT_DIR, $CANDGEN_ROOT,
    текущая папка и её родители, вложенные папки (до 3 уровней)."""
    here = Path.cwd(); ok = lambda p: (p / 'train.parquet').exists() and (p / 'benchmark_items.parquet').exists()
    for p in [Path(x) for x in (PROJECT_DIR, os.environ.get('CANDGEN_ROOT')) if x] + [here, *here.parents]:
        if ok(p): return p.resolve()
    for pat in ('*/train.parquet', '*/*/train.parquet', '*/*/*/train.parquet'):
        for h in sorted(here.glob(pat)):
            if ok(h.parent): return h.parent.resolve()
    raise FileNotFoundError(f'Не найдена папка с train.parquet и benchmark_items.parquet (текущая папка: {here}; в ней: '
                            f'{sorted(x.name for x in here.iterdir())[:30]}). Положите parquet-файлы рядом с ноутбуком или задайте PROJECT_DIR.')
ROOT = find_root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT / 'src'))
_missing = [f for f in ('train.parquet', 'benchmark_queries.parquet', 'benchmark_items.parquet') if not (ROOT / f).exists()]
if _missing: raise FileNotFoundError(f'В {ROOT} нет {_missing}: положите их рядом с ноутбуком')
for d in ('data', 'models', 'subs', 'results'): os.makedirs(d, exist_ok=True)
print('project root:', ROOT, '| перенесённые модели:', sorted(x.name for x in Path('models').iterdir() if x.is_dir()))
import shutil
def disk_free_gb(): return shutil.disk_usage(ROOT).free / 1e9
print(f'свободно на диске: {disk_free_gb():.1f} ГБ')
def _du(p):
    p = Path(p).expanduser()
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file() and not f.is_symlink()) / 1e9 if p.exists() else 0.0
print('  занято: models/ {:.1f} ГБ, data/ {:.1f} ГБ, кеш HF hub {:.1f} ГБ, кеш hf_xet {:.1f} ГБ'.format(
      _du('models'), _du('data'), _du('~/.cache/huggingface/hub'), _du('~/.cache/huggingface/xet')))
if disk_free_gb() < 40: print('ВНИМАНИЕ: для полного прогона нужно ~40 ГБ свободного места (README.md, раздел «Воспроизведение»)')

SMOKE = False            # True: быстрый прогон кода на крошечных подвыборках

ENCODERS = {
    # pool: mean | cls | last (last-token, левый паддинг); qlen: длина запроса в токенах; ebs: батч при кодировании
    # GPU 24–40 ГБ: уменьшите bs у e5l/ubge/qwen06 до 64–128
    'e5s':    dict(name='intfloat/multilingual-e5-small', pool='mean', qp='query: ', dp='passage: ', lr=5e-5, bs=64, gc=False, qlen=32, ebs=1024),   # batch 64 — рецепт, которым обучены e5s_T/e5s_F финала
    'e5b':    dict(name='intfloat/multilingual-e5-base',  pool='mean', qp='query: ', dp='passage: ', lr=3e-5, bs=256, gc=False, qlen=32, ebs=512),
    'e5l':    dict(name='intfloat/multilingual-e5-large', pool='mean', qp='query: ', dp='passage: ', lr=2e-5, bs=256, gc=True,  qlen=32, ebs=256),
    'ubge':   dict(name='deepvk/USER-bge-m3',             pool='cls',  qp='',        dp='',          lr=2e-5, bs=256, gc=True,  qlen=32, ebs=256),
    'qwen06': dict(name='Qwen/Qwen3-Embedding-0.6B',      pool='last',
                   qp='Instruct: Given a search query on a services marketplace, retrieve the service listing that the user will choose\nQuery: ',
                   dp='', lr=2e-5, bs=128, gc=True, qlen=64, ebs=256),
}
SCALE_TEST = ['e5s', 'e5b', 'e5l', 'ubge', 'qwen06']   # e5s — база (тот же рецепт, что дал LB 0.916)
BI_EPOCHS = 1
DLEN = 128               # длина текста объявления в токенах
SCALE = 20.0             # 1/temperature в InfoNCE
PAIR_CAP = 20            # не более N пар на один текст запроса (ближе к хвостовому распределению бенчмарка)
HARDNEG = True           # второй раунд дообучения лучшего энкодера с hard negatives
HN_RANKS = (3, 30)       # откуда брать hard negative в выдаче майнинга (топ-3 пропускаем: там много «ложных негативов»)
NRK = 15000              # запросов для обучения ранкера
N_JOBS = min(16, os.cpu_count() or 1)   # процессы для сборки признаков (fork, Linux)
LGB_SEEDS = 3
LGB_ROUNDS = 400

CE_MODEL = 'BAAI/bge-reranker-v2-m3'   # любой AutoModelForSequenceClassification-кросс-энкодер
CE_TOPK = 100            # сколько кандидатов первого прохода переоценивает cross-encoder
CE_NEG = 7               # негативов на группу (листовой softmax 1+7)
CE_MAX_GROUPS = 80000
CE_EPOCHS = 1
CE_BS_GROUPS = 16        # групп в батче -> 16*(1+7)=128 пар
CE_LR = 1e-5
CE_LEN = 256
CE_GC = True
CE_FULL = True           # для бенчмарка дообучить cross-encoder на всём train (иначе — версия на T)
R0 = 15                  # слоты под объявления вне 114 для запросов с search_category=0
MIN_GAIN = 0.003         # порог «берём улучшение» по R@35w
CLEANUP = True           # удалять промежуточные модели/признаки, которые больше не понадобятся (экономия диска)

if SMOKE:
    ENCODERS['e5s_cls'] = dict(ENCODERS['e5s'], pool='cls')
    ENCODERS['e5s_last'] = dict(ENCODERS['e5s'], pool='last')
    for k in ENCODERS: ENCODERS[k].update(bs=32, ebs=256)
    SCALE_TEST = ['e5s', 'e5s_cls', 'e5s_last']
    NRK = 600; N_JOBS = 1; LGB_SEEDS = 1; LGB_ROUNDS = 50
    CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'; CE_TOPK = 30; CE_MAX_GROUPS = 400; CE_BS_GROUPS = 4; CE_GC = False; CE_LR = 3e-5
    SMOKE_PAIRS = 3000; SMOKE_CORPUS = 20000
TAG = 'smoke_' if SMOKE else ''
print('SMOKE =', SMOKE)

## Зависимости
Недостающие пакеты доустанавливаются в текущее окружение; `torch` нужно установить заранее под свою версию CUDA.

In [ ]:
assert Path('src/pipeline.py').exists(), 'нет папки src/: запускайте ноутбук из корня репозитория'
# зависимости: недостающее доустанавливается в текущее окружение (torch должен быть установлен заранее, под CUDA сервера)
import importlib.util
from importlib.metadata import version as _version
def _ver(pkg):
    try: return tuple(int(x) for x in re.findall(r'\d+', _version(pkg))[:2])
    except Exception: return None
_need = [pkg for mod, pkg in [('lightgbm', 'lightgbm'), ('Stemmer', 'pystemmer'), ('sklearn', 'scikit-learn'), ('scipy', 'scipy'),
                              ('pyarrow', 'pyarrow'), ('threadpoolctl', 'threadpoolctl')] if importlib.util.find_spec(mod) is None]
if _ver('transformers') is None or _ver('transformers') < (4, 51): _need.append('transformers>=4.51')   # Qwen3 требует >= 4.51
assert importlib.util.find_spec('torch') is not None, 'нет torch: установите его под CUDA сервера (README.md, раздел «Воспроизведение»)'
if _need:
    print('доустанавливаю:', _need); subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_need], check=True)
print('зависимости ок; transformers', _version('transformers'), '| torch', _version('torch'))

## 0. Данные
Если промежуточных артефактов нет, они строятся скриптами из `scripts/`:
- `prep.py` — единая таблица объявлений и лог;
- `build_tok.py` — стемминг;
- `item_text.py` — компактный текст объявления;
- `split.py` — псевдо-бенчмарк: объявления делятся по хешу `item_id` на корпус V (178k) и лог T, 3000 val-запросов и 45k запросов для ранкера.

In [ ]:
def run(cmd):
    print('$', cmd, flush=True); subprocess.run(cmd, shell=True, check=True)

need = ['data/items.parquet', 'data/log.parquet', 'data/bench_q.parquet', 'data/item_toks.pkl', 'data/item_text.parquet',
        'data/T_log.parquet', 'data/val_q.parquet', 'data/rk_q.parquet', 'data/val_corpus.npy']
if not all(Path(p).exists() for p in need):
    for s in ['prep.py', 'build_tok.py', 'item_text.py', 'split.py']:
        run(f'{sys.executable} scripts/{s}')

import lightgbm as lgb          # до torch: иначе на macOS конфликт libomp (segfault)
import torch, torch.nn.functional as F
if sys.platform == 'darwin': torch.set_num_threads(1)   # macOS: иначе CPU-операции torch рядом с lightgbm падают (libomp)
from threadpoolctl import threadpool_limits
from common import toks, norm, param_dict
from pipeline import Builder, add_ranks

DEV = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
BF16 = DEV == 'cuda' and torch.cuda.is_bf16_supported()
torch.manual_seed(0); np.random.seed(0); random.seed(0)
if DEV == 'cuda': torch.backends.cuda.matmul.allow_tf32 = True
print('device', DEV, 'bf16', BF16, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

ITEMS = pd.read_parquet('data/items.parquet', columns=['iid', 'item_id', 'in_corpus', 'item_location_id', 'item_category_id'])
ITEXT = pd.read_parquet('data/item_text.parquet').text.values
ILOC = ITEMS.item_location_id.values
LOG = pd.read_parquet('data/log.parquet'); TLOG = pd.read_parquet('data/T_log.parquet')
VAL = pd.read_parquet('data/val_q.parquet'); RK = pd.read_parquet('data/rk_q.parquet').iloc[:NRK].reset_index(drop=True)
BQ = pd.read_parquet('data/bench_q.parquet')
VCORP = np.load('data/val_corpus.npy'); BCORP = np.sort(ITEMS.iid.values[ITEMS.in_corpus.values == 1])
TCORP = np.unique(TLOG.iid.values); FCORP = np.unique(LOG.iid.values)       # корпуса для майнинга негативов

if SMOKE:
    rs = np.random.default_rng(0)
    VAL = VAL.iloc[:200].reset_index(drop=True); RK = RK.iloc[:NRK].reset_index(drop=True); BQ = BQ.iloc[:150].reset_index(drop=True)
    rel_items = np.unique(np.concatenate([np.concatenate(VAL.rel.values), np.concatenate(RK.rel.values)]))
    VCORP = np.unique(np.r_[rel_items, rs.choice(VCORP, SMOKE_CORPUS, replace=False)])
    BCORP = np.sort(rs.choice(BCORP, SMOKE_CORPUS, replace=False))
    TCORP = np.sort(rs.choice(TCORP, SMOKE_CORPUS, replace=False)); FCORP = np.sort(rs.choice(FCORP, SMOKE_CORPUS, replace=False))
print(f'T log {len(TLOG)}, full log {len(LOG)}, val {len(VAL)}, rk {len(RK)}, bench {len(BQ)}; corpora V {len(VCORP)} B {len(BCORP)} T {len(TCORP)} F {len(FCORP)}')

def qtext(q, f): return q if not f else f'{q} | {f}'
VAL_QT = [qtext(a, b) for a, b in zip(VAL.search_query, VAL.search_infm_params_text)]
RK_QT = [qtext(a, b) for a, b in zip(RK.search_query, RK.search_infm_params_text)]
BQ_QT = [qtext(a, b) for a, b in zip(BQ.search_query, BQ.search_infm_params_text)]

## 1. Метрики и приор локации
`W_VAL` перевзвешивает val-запросы под состав бенчмарка. Сегменты: Москва, СПб, их регионы, «вся Россия», прочие; с фильтрами или без.

In [ ]:
def seg(df):
    big = df.search_location_id.map({637640: 'MSK', 653240: 'SPB', 107620: 'MSKreg', 107621: 'SPBreg', 621540: 'RU'}).fillna('other')
    return big + '|' + np.where(df.search_infm_params_text == '', 'nof', 'f')
_sv, _sb = seg(VAL), seg(pd.read_parquet('data/bench_q.parquet'))
W_VAL = (_sb.value_counts(normalize=True) / _sv.value_counts(normalize=True)).reindex(_sv).fillna(0).values
NREL_VAL = VAL.rel.str.len().values

def eval_rank(df, score, qdf=VAL, w=None):
    """df: строки кандидатов (qi, y); score: массив скоров. Позитивы вне df считаются промахом."""
    w = W_VAL if w is None else w
    nrel = qdf.rel.str.len().values
    d = pd.DataFrame({'qi': df.qi.values, 'y': df.y.values, 's': score})
    d['r'] = d.groupby('qi').s.rank(ascending=False, method='first')
    out = {}
    for k in (50, 35):
        hit = d[(d.r <= k) & (d.y == 1)].groupby('qi').size().reindex(range(len(qdf))).fillna(0).values / nrel
        out[f'R@{k}'] = hit.mean(); out[f'R@{k}w'] = np.average(hit, weights=w)
    return out

RESULTS = []
def log_result(stage, name, m):
    RESULTS.append(dict(stage=stage, name=name, **{k: round(float(v), 4) for k, v in m.items()}))
    print(f'[{stage}] {name}:', {k: round(float(v), 4) for k, v in m.items()}, flush=True)
    pd.DataFrame(RESULTS).to_csv(f'results/{TAG}validation_metrics.csv', index=False)

class LocPrior:
    """P(локация объявления | локация поиска) по логу, со сглаживанием к глобальному распределению."""
    def __init__(self, log, alpha=5.0):
        il = ILOC[log.iid.values]
        self.pair = pd.DataFrame({'s': log.search_location_id.values, 'i': il}).groupby(['s', 'i']).size()
        self.N = log.groupby('search_location_id').size(); self.g = pd.Series(il).value_counts(normalize=True); self.alpha = alpha
    def logp(self, sloc, cand_locs):
        cl = pd.Series(cand_locs); base = cl.map(self.g).fillna(1e-6).values; Ns = self.N.get(sloc, 0)
        if Ns:
            p = (cl.map(self.pair[sloc]).fillna(0).values + self.alpha * base) / (Ns + self.alpha)
        else:
            p = np.where(cand_locs == sloc, 0.77, 0.23 * base)
        return np.log(p + 1e-7).astype(np.float32)
LP_T, LP_F = LocPrior(TLOG), LocPrior(LOG)
REF_STD = 0.108

## 2. Bi-encoder: обёртка, дообучение, кодирование
- **Функция потерь:** симметричный InfoNCE (температура 0.05), in-batch негативы.
- **Батч:** без повторов текстов запросов и объявлений, чтобы не было ложных негативов.
- **Второй раунд:** к каждой паре добавляется hard negative.
- **Пулинг:** mean, CLS или last-token (для Qwen3-Embedding: левый паддинг и гарантированный EOS в конце).

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from safetensors import safe_open
from safetensors.torch import load_file as st_load, save_file as st_save

def model_complete(d):
    """config + все *.safetensors читаются целиком (обрезанный при сбое файл не пройдёт)."""
    d = Path(d); st = list(d.glob('*.safetensors'))
    if not (d / 'config.json').exists() or not st: return False
    try:
        for f in st:
            with safe_open(str(f), 'pt') as fh: fh.keys()
        return True
    except Exception:
        return False

def prepare_out(out_dir):
    """True — модель уже есть целиком; неполную папку (после сбоя) удаляем, чтобы освободить место и переобучить."""
    if model_complete(out_dir): print('skip, exists:', out_dir); return True
    if Path(out_dir).exists(): print('неполная модель после сбоя, удаляю и переобучаю:', out_dir); shutil.rmtree(out_dir)
    return False

def save_bf16(model, tok, out_dir):
    model.to(torch.bfloat16).save_pretrained(out_dir); tok.save_pretrained(out_dir)

PROTECTED = {'e5s_T', 'e5s_F'}   # перенесённые модели: их SHA проверяется, не трогаем
_COMPACT_PY = r"""
import sys, os, torch
from safetensors import safe_open
from safetensors.torch import load_file, save_file
f = sys.argv[1]
with safe_open(f, 'pt') as fh: meta = fh.metadata()
t = {k: (v.to(torch.bfloat16) if v.dtype == torch.float32 else v.clone()) for k, v in load_file(f).items()}
save_file(t, f + '.tmp', metadata=meta or {'format': 'pt'}); os.replace(f + '.tmp', f)
"""
def purge_incomplete():
    """Удаляет неполные модели (обрезанные при сбое записи) — освобождает место; они будут переобучены."""
    for d in sorted(Path('models').glob('*')):
        if d.is_dir() and not model_complete(d): print('  удаляю неполную модель:', d); shutil.rmtree(d)

def compact_fp32_models():
    """Переписывает fp32-веса уже сохранённых моделей в bf16 (вдвое меньше на диске). Запись — во временный файл
    с атомарной подменой: исходный файл отображён в память (mmap), перезаписывать его на месте нельзя."""
    purge_incomplete()
    for d in sorted(Path('models').glob('*')):
        if not d.is_dir() or d.name in PROTECTED or not model_complete(d): continue
        for f in d.glob('*.safetensors'):
            with safe_open(str(f), 'pt') as fh:
                if not any(fh.get_slice(k).get_dtype() == 'F32' for k in fh.keys()): continue
                meta = fh.metadata()
            # отдельный процесс: на macOS CPU-операции torch в одном процессе с lightgbm падают (конфликт libomp)
            r = subprocess.run([sys.executable, '-c', _COMPACT_PY, str(f)], capture_output=True, text=True)
            if r.returncode == 0: print('  bf16:', f)
            else:
                Path(str(f) + '.tmp').unlink(missing_ok=True)
                print('  не удалось сжать (скорее всего, не хватило места):', f, r.stderr.strip()[-200:])

def drop_hf_cache(names):
    try:
        from huggingface_hub import scan_cache_dir
        info = scan_cache_dir(); revs = [r.commit_hash for repo in info.repos if repo.repo_id in names for r in repo.revisions]
        if revs: st_ = info.delete_revisions(*revs); print(f'  HF-кеш: освобождаю {st_.expected_freed_size / 1e9:.1f} ГБ'); st_.execute()
    except Exception as e: print('  HF-кеш не очищен:', e)

class Encoder:
    def __init__(self, cfg, path=None):
        self.cfg = cfg; src = path or cfg['name']
        self.tok = AutoTokenizer.from_pretrained(src)
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        if cfg['pool'] == 'last': self.tok.padding_side = 'left'
        self.model = AutoModel.from_pretrained(src).to(DEV).float()   # веса на диске в bf16; в fp32 переводим уже на GPU
        self.eos_fix = cfg['pool'] == 'last' and self.tok('тест')['input_ids'][-1] != self.tok.eos_token_id
    def _batch(self, texts, L):
        if not self.eos_fix:
            return self.tok(texts, padding=True, truncation=True, max_length=L, return_tensors='pt').to(DEV)
        e = self.tok(texts, truncation=True, max_length=L - 1)
        e = {'input_ids': [x + [self.tok.eos_token_id] for x in e['input_ids']], 'attention_mask': [m + [1] for m in e['attention_mask']]}
        return self.tok.pad(e, padding=True, return_tensors='pt').to(DEV)
    def embed(self, texts, L):
        b = self._batch(texts, L)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=BF16):
            h = self.model(**b).last_hidden_state
        h = h.float(); m = b['attention_mask'].unsqueeze(-1).float(); p = self.cfg['pool']
        e = (h * m).sum(1) / m.sum(1) if p == 'mean' else (h[:, 0] if p == 'cls' else h[:, -1])
        return F.normalize(e, dim=-1)
    @torch.no_grad()
    def encode(self, texts, L, bs=None):
        self.model.eval(); bs = bs or self.cfg['ebs']
        order = np.argsort([len(t) for t in texts]); out = []
        for i in range(0, len(texts), bs):
            out.append(self.embed([texts[j] for j in order[i:i + bs]], L).cpu().numpy().astype(np.float16))
        E = np.concatenate(out); R = np.empty_like(E); R[order] = E
        return R
    def q(self, texts): return [self.cfg['qp'] + t for t in texts]
    def d(self, iids): return [self.cfg['dp'] + ITEXT[i] for i in iids]
    def free(self):
        del self.model; gc.collect()
        if DEV == 'cuda': torch.cuda.empty_cache()

def make_pairs(log, seed=0):
    p = pd.DataFrame({'qt': [qtext(a, b) for a, b in zip(log.search_query, log.search_infm_params_text)],
                      'q': log.search_query.values, 'sloc': log.search_location_id.values, 'iid': log.iid.values})
    p = p.drop_duplicates(['qt', 'iid']).sample(frac=1, random_state=seed).groupby('q').head(PAIR_CAP).reset_index(drop=True)
    if SMOKE: p = p.iloc[:SMOKE_PAIRS].reset_index(drop=True)
    return p
PAIRS_T, PAIRS_F = make_pairs(TLOG), make_pairs(LOG)
print('pairs T', len(PAIRS_T), 'F', len(PAIRS_F))

def train_biencoder(key, pairs, out_dir, init=None, hardneg=None, epochs=BI_EPOCHS):
    """hardneg: массив iid (по одному на пару) или None."""
    if prepare_out(out_dir): return out_dir
    cfg = ENCODERS[key]; enc = Encoder(cfg, init); model = enc.model; model.train()
    if cfg['gc']:
        model.gradient_checkpointing_enable()
        if hasattr(model.config, 'use_cache'): model.config.use_cache = False
    bs = cfg['bs']; nb = max(1, int(len(pairs) / bs * epochs))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, s / max(1, 0.05 * nb)) * max(0.0, (nb - s) / max(1, 0.95 * nb)))
    qts = enc.q(pairs.qt.tolist()); qv, iv = pairs.q.values, pairs.iid.values
    hn = None if hardneg is None else np.asarray(hardneg)
    rng = np.random.default_rng(0); step, t0, lsum = 0, time.time(), 0.0
    while step < nb:
        cur, sq, sd = [], set(), set()
        for i in rng.permutation(len(pairs)):
            if qv[i] in sq or iv[i] in sd or (hn is not None and (hn[i] in sd or hn[i] == iv[i])): continue
            cur.append(i); sq.add(qv[i]); sd.add(iv[i])
            if hn is not None: sd.add(hn[i])
            if len(cur) < bs: continue
            qe = enc.embed([qts[j] for j in cur], cfg['qlen'])
            docs = list(iv[cur]) + ([] if hn is None else list(hn[cur]))
            de = enc.embed(enc.d(docs), DLEN)
            s = qe @ de.T * SCALE; lab = torch.arange(len(cur), device=DEV)
            loss = (F.cross_entropy(s, lab) + F.cross_entropy(s[:, :len(cur)].T, lab)) / 2
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            step += 1; lsum += loss.item(); cur, sq, sd = [], set(), set()
            if step % 100 == 0 or step == nb:
                print(f'  {key} step {step}/{nb} loss {lsum / (100 if step % 100 == 0 else step % 100):.4f} {time.time() - t0:.0f}s', flush=True); lsum = 0.0
            if step >= nb: break
    save_bf16(model, enc.tok, out_dir); enc.free()
    return out_dir

def emb_path(tag, name): return f'data/{TAG}emb_{tag}_{name}.npy'
def encode_sets(key, model_dir, tag, sets):
    """sets: список из valcorpus, benchcorpus, valQ, rkQ, benchQ, valQp, rkQp, benchQp, logQT, logQF."""
    todo = [s for s in sets if not Path(emb_path(tag, s)).exists()]
    if not todo: return
    cfg = ENCODERS[key]; enc = Encoder(cfg, model_dir); t0 = time.time()
    for s in todo:
        if s == 'valcorpus': E = enc.encode(enc.d(VCORP), DLEN)
        elif s == 'benchcorpus': E = enc.encode(enc.d(BCORP), DLEN)
        elif s in ('valQ', 'rkQ', 'benchQ'): E = enc.encode(enc.q({'valQ': VAL_QT, 'rkQ': RK_QT, 'benchQ': BQ_QT}[s]), cfg['qlen'])
        elif s in ('valQp', 'rkQp', 'benchQp'): E = enc.encode(enc.q(list({'valQp': VAL, 'rkQp': RK, 'benchQp': BQ}[s].search_query)), cfg['qlen'])
        elif s in ('logQT', 'logQF'): E = enc.encode(enc.q(list(np.load(f'data/{s}_texts.npy', allow_pickle=True))), cfg['qlen'])
        np.save(emb_path(tag, s), E); print(f'  encoded {tag}/{s} {E.shape} {time.time() - t0:.0f}s', flush=True)
    enc.free()

for s, lg in (('logQT', TLOG), ('logQF', LOG)):
    if not Path(f'data/{s}_texts.npy').exists():
        np.save(f'data/{s}_texts.npy', np.array(sorted(lg.search_query.unique()), dtype=object))

## 2.1. Перенесённые модели
Если `models/e5s_T` и `models/e5s_F` скопированы с локальной машины, ноутбук их не переобучает: это ровно те энкодеры, что дали LB 0.916, поэтому база `P0_e5s` получается точной. Ячейка проверяет:
- **целостность копирования** — SHA-256 весов;
- **поведение на сервере** — косинусы на трёх фиксированных парах «запрос–объявление» против эталона с локальной машины (допуск 0.03 на bf16/CUDA).

Если проверка не прошла, скопируйте папку заново или удалите её: тогда модель будет обучена на сервере (около 5–10 минут). Файлы `models/lgb_*.txt` ноутбуком не используются.

In [ ]:
import hashlib
REF_MODELS = {"e5s_T": {"sims": [[0.7054, -0.031, 0.046], [-0.0644, 0.7, -0.1461], [0.0272, -0.1065, 0.7387]], "sha16": "2de8ec7228282f64"}, "e5s_F": {"sims": [[0.7074, -0.0657, 0.0516], [-0.0903, 0.6587, -0.1083], [0.0387, -0.073, 0.6844]], "sha16": "9e5446b1fa6fc3ef"}}
REF_Q = ['query: монтаж видеодомофонов', 'query: автоподбор | Рейтинг пользователя 4 звезды и выше', 'query: баня на дровах']
REF_D = ['passage: Установка домофонов и видеонаблюдения. Монтаж и установка техники. Выезд мастера, гарантия',
         'passage: Автоподбор. Разовый осмотр автомобиля. Автосервис, аренда. Проверка авто перед покупкой',
         'passage: Баня на дровах с купелью. Праздники, мероприятия. Почасовая аренда, парная на дровах']
for name, ref in REF_MODELS.items():
    d = Path('models') / name
    if not model_complete(d):
        print(f'{name}: нет в models/ (или файл неполный) — будет обучена на сервере'); continue
    h = hashlib.sha256()
    with open(d / 'model.safetensors', 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b''): h.update(chunk)
    assert h.hexdigest()[:16] == ref['sha16'], f'{name}: SHA-256 весов не совпал — файл повреждён при копировании, перенесите заново'
    enc = Encoder(ENCODERS['e5s'], str(d))
    with torch.no_grad():
        S = (enc.embed(REF_Q, 32) @ enc.embed(REF_D, 128).T).cpu().numpy()
    enc.free()
    diff = np.abs(S - np.array(ref['sims'])).max()
    print(f'{name}: SHA ok, max |Δcos| = {diff:.4f}, диагональ {np.round(np.diag(S), 3)}')
    assert diff < 0.03, f'{name}: поведение модели на сервере расходится с локальным (Δ={diff:.3f}) — проверьте версии transformers/torch'
print('перенесённые модели в порядке')

## 3. Тест энкодеров на масштаб (dense-only)
Для каждого энкодера:
1. дообучение на T-логе (1 эпоха);
2. кодирование val-корпуса и val-запросов;
3. полнота dense-поиска: глобально (`dR@50`), с приором локации (`dR@50loc`), и перевзвешенная (`dR@35wloc`).

Выбор лучшего — по `dR@35wloc`. Правило: если small → large даёт меньше +1 п.п. dense-only, масштаб не узкое место, и основной прирост ждём от cross-encoder.

In [ ]:
def dense_eval(tag, qdf=VAL, corpus=VCORP, lp=LP_T, ws=(0.02,)):
    D = torch.tensor(np.load(emb_path(tag, 'valcorpus')), device=DEV).float()
    Q = torch.tensor(np.load(emb_path(tag, 'valQ')), device=DEV).float()
    cpos = pd.Series(np.arange(len(corpus)), index=corpus)
    rels = [set(cpos[r].values) for r in qdf.rel]
    cl = ILOC[corpus]; hits = {'glob': np.zeros(len(qdf)), **{w: np.zeros(len(qdf)) for w in ws}}; hits35 = {w: np.zeros(len(qdf)) for w in ws}
    for sloc, g in qdf.groupby('search_location_id'):
        L = torch.tensor(lp.logp(sloc, cl), device=DEV)
        for ch in np.array_split(g.index.values, max(1, len(g) // 256)):
            S = Q[ch] @ D.T
            top = torch.topk(S, 50, dim=1).indices.cpu().numpy()
            for i, t in zip(ch, top): hits['glob'][i] = len(rels[i] & set(t)) / len(rels[i])
            Sn = S / S.std(1, keepdim=True) * REF_STD
            for w in ws:
                top = torch.topk(Sn + w * L, 50, dim=1).indices.cpu().numpy()
                for i, t in zip(ch, top):
                    hits[w][i] = len(rels[i] & set(t)) / len(rels[i]); hits35[w][i] = len(rels[i] & set(t[:35])) / len(rels[i])
    w0 = ws[0]
    return {'dR@50': hits['glob'].mean(), 'dR@50loc': hits[w0].mean(), 'dR@35wloc': np.average(hits35[w0], weights=W_VAL)}

compact_fp32_models(); print(f'свободно: {disk_free_gb():.1f} ГБ')
DENSE = {}
for key in SCALE_TEST:
    t0 = time.time()
    mdir = train_biencoder(key, PAIRS_T, f'models/{TAG}{key}_T')
    encode_sets(key, mdir, f'{key}T', ['valcorpus', 'valQ'])
    DENSE[key] = dense_eval(f'{key}T'); log_result('dense', f'{key}_T', DENSE[key])
    print(f'  {key}: {time.time() - t0:.0f}s total', flush=True)
BEST = max(DENSE, key=lambda k: DENSE[k]['dR@35wloc'])
print('BEST encoder by dR@35wloc:', BEST)
if CLEANUP:   # дальше нужны только лучший энкодер и e5s (база и второй dense-признак)
    for k in SCALE_TEST:
        if k not in (BEST, 'e5s') and Path(f'models/{TAG}{k}_T').exists(): shutil.rmtree(f'models/{TAG}{k}_T'); print('  удалена', f'models/{TAG}{k}_T')
    drop_hf_cache({ENCODERS[k]['name'] for k in SCALE_TEST if k not in (BEST, 'e5s')} - {ENCODERS[BEST]['name'], ENCODERS['e5s']['name']})
    print(f'свободно: {disk_free_gb():.1f} ГБ')

## 4. Hard negatives: майнинг и второй раунд
Порядок майнинга:
1. лучший энкодер T-версии кодирует объявления T-лога;
2. для каждого (запрос, локация) берётся топ-60 по «dense (нормированный) + 0.02·log P(локация)»;
3. из выдачи исключаются все объявления, выбранные по тому же тексту запроса.

Для второго раунда bi-encoder берётся негатив с позиций `HN_RANKS`. Та же выдача потом служит источником групп для cross-encoder.

In [ ]:
def mine(key, model_dir, pairs, log, corpus, lp, topn=60, cache=None):
    """-> list (по парам) массивов iid-кандидатов (без выбранных по этому тексту), упорядоченных по убыванию скоринга."""
    if cache and Path(cache).exists(): return pickle.load(open(cache, 'rb'))
    cfg = ENCODERS[key]; enc = Encoder(cfg, model_dir)
    D = torch.tensor(enc.encode(enc.d(corpus), DLEN), device=DEV).float()
    uq = pairs[['qt', 'sloc']].drop_duplicates().reset_index(drop=True)
    Q = torch.tensor(enc.encode(enc.q(uq.qt.tolist()), cfg['qlen']), device=DEV).float(); enc.free()
    cl = ILOC[corpus]; res = [None] * len(uq); t0 = time.time()
    for sloc, g in uq.groupby('sloc'):
        L = torch.tensor(lp.logp(sloc, cl), device=DEV)
        for ch in np.array_split(g.index.values, max(1, len(g) // 512)):
            S = Q[ch] @ D.T; S = S / S.std(1, keepdim=True) * REF_STD + 0.02 * L
            top = torch.topk(S, min(topn, len(corpus)), dim=1).indices.cpu().numpy()
            for i, t in zip(ch, top): res[i] = corpus[t]
    chosen = log.groupby('search_query').iid.apply(set)
    key2 = {(a, b): i for i, (a, b) in enumerate(zip(uq.qt, uq.sloc))}
    out = []
    for qt, q, sl in zip(pairs.qt, pairs.q, pairs.sloc):
        ch_ = chosen.get(q, set()); c = res[key2[(qt, sl)]]
        out.append(np.array([x for x in c if x not in ch_], dtype=np.int64))
    print(f'  mined {len(out)} lists in {time.time() - t0:.0f}s', flush=True)
    del D, Q; gc.collect()
    if cache: pickle.dump(out, open(cache, 'wb'))
    return out

def pick_hardneg(cands, corpus, lo_hi=HN_RANKS, seed=0):
    rng = np.random.default_rng(seed); out = np.empty(len(cands), np.int64)
    for i, c in enumerate(cands):
        pool = c[lo_hi[0]:lo_hi[1]]
        out[i] = rng.choice(pool) if len(pool) else rng.choice(corpus)
    return out

MINED_T = mine(BEST, f'models/{TAG}{BEST}_T', PAIRS_T, TLOG, TCORP, LP_T, cache=f'data/{TAG}mined_T_{BEST}.pkl')
HN_KEY = None
if HARDNEG:
    hn = pick_hardneg(MINED_T, TCORP)
    mdir = train_biencoder(BEST, PAIRS_T, f'models/{TAG}{BEST}_T_hn', init=f'models/{TAG}{BEST}_T', hardneg=hn)
    encode_sets(BEST, mdir, f'{BEST}Thn', ['valcorpus', 'valQ'])
    m = dense_eval(f'{BEST}Thn'); log_result('dense', f'{BEST}_T_hardneg', m)
    if m['dR@35wloc'] > DENSE[BEST]['dR@35wloc'] + 0.002: HN_KEY = BEST
print('use hard-negative round for', HN_KEY)

## 5. Полный пайплайн с новым энкодером
Кандидаты и признаки собираются тем же `Builder`, что дал 0.945 / 0.916. Кандидаты: BM25, dense (с приором локации и глобально), гибрид, символьные n-граммы, микрокатегории, kNN по логу; признаки — около 60 штук. Сравниваются три варианта:
- `P0` — база: основной dense-сигнал e5-small;
- `P1` — лучший энкодер как основной;
- `P2` — лучший энкодер плюс e5-small вторым dense-признаком.

Ранкер — ансамбль LightGBM LambdaRank по `LGB_SEEDS` сидам. Сборка признаков распараллелена через fork.

In [ ]:
import multiprocessing as mp
_PB = {}
def _pb_worker(rng_):
    lo, hi = rng_; B, qdf, dense, qplain, labels = _PB['args']
    with threadpool_limits(1):
        Fr = B.build(qdf.iloc[lo:hi], {k: (Q[lo:hi], D) for k, (Q, D) in dense.items()}, with_labels=labels,
                     qplain=None if qplain is None else qplain[lo:hi])
    Fr['qi'] += lo
    return Fr
def par_build(B, qdf, dense, qplain, labels=True, chunk=250):
    qdf = qdf.reset_index(drop=True); _PB['args'] = (B, qdf, dense, qplain, labels)
    chunks = [(i, min(len(qdf), i + chunk)) for i in range(0, len(qdf), chunk)]
    if N_JOBS <= 1: parts = [_pb_worker(c) for c in chunks]
    else:
        with mp.get_context('fork').Pool(N_JOBS) as pool: parts = pool.map(_pb_worker, chunks)
    Fr = pd.concat(parts, ignore_index=True)
    for c in Fr.columns:
        if Fr[c].dtype == np.float64: Fr[c] = Fr[c].astype(np.float32)
    return Fr

def pipe_encode(key, tag, model_dir, bench=False):
    sets = ['valcorpus', 'valQ', 'rkQ', 'valQp', 'rkQp', 'logQT'] if not bench else ['benchcorpus', 'benchQ', 'benchQp', 'logQF']
    encode_sets(key, model_dir, tag, sets)

def load_dense(tag, split):
    corp = 'benchcorpus' if split == 'bench' else 'valcorpus'
    return np.load(emb_path(tag, f'{split}Q')).astype(np.float32), np.load(emb_path(tag, corp)).astype(np.float32)

def build_pipeline_feats(name, d1, d2=None):
    """d1/d2: теги эмбеддингов (d1 — основной, он же для kNN по логу). Возвращает (F_val, F_rk)."""
    fv, fr = f'data/{TAG}feats_{name}_val.parquet', f'data/{TAG}feats_{name}_rk.parquet'
    if Path(fv).exists() and Path(fr).exists(): return pd.read_parquet(fv), pd.read_parquet(fr)
    knn = (np.load('data/logQT_texts.npy', allow_pickle=True), np.load(emb_path(d1, 'logQT')))
    t0 = time.time(); B = Builder(VCORP, TLOG, knn=knn)
    out = []
    for split, qdf in (('val', VAL), ('rk', RK)):
        dense = {'d1': load_dense(d1, split)}
        if d2: dense['d2'] = load_dense(d2, split)
        Fr = par_build(B, qdf, dense, np.load(emb_path(d1, f'{split}Qp')).astype(np.float32))
        Fr.to_parquet(fv if split == 'val' else fr, index=False); out.append(Fr)
        rec = (Fr.groupby('qi').y.sum().reindex(range(len(qdf))).fillna(0).values / qdf.rel.str.len().values).mean()
        print(f'  {name}/{split}: {len(Fr)} rows, {len(Fr) / len(qdf):.0f} cands/q, candidate recall {rec:.4f}, {time.time() - t0:.0f}s', flush=True)
    del B; gc.collect()
    return out[0], out[1]

LGB_P = dict(objective='lambdarank', learning_rate=0.05, num_leaves=63, min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8,
             bagging_freq=1, lambdarank_truncation_level=60, verbose=-1, num_threads=os.cpu_count())
def feat_cols(Fr, extra_drop=()): return [c for c in Fr.columns if c not in ('qi', 'ci', 'y', 'i_cat114', *extra_drop)]
def lgb_fit(Fr, feats, seeds=LGB_SEEDS, rounds=LGB_ROUNDS):
    Fr = Fr[Fr.groupby('qi').y.transform('max') > 0].sort_values('qi', kind='stable')
    ds = lgb.Dataset(Fr[feats], Fr.y, group=Fr.groupby('qi', sort=True).size().values, free_raw_data=False)
    return [lgb.train(dict(LGB_P, seed=s), ds, rounds) for s in range(seeds)]
def lgb_pred(models, Fr, feats): return np.mean([m.predict(Fr[feats]) for m in models], axis=0)

def enc_ref(key, hn):
    return (f'{key}Thn', f'models/{TAG}{key}_T_hn') if hn else (f'{key}T', f'models/{TAG}{key}_T')
def tags_of(P): return P['tags'][0], (P['tags'][1] if len(P['tags']) > 1 else None)

PIPE = {}
variants = {'P0_e5s': [('e5s', False)]}                       # база: тот же рецепт, что LB 0.916
if BEST != 'e5s' or HN_KEY == 'e5s': variants[f'P1_{BEST}' + ('_hn' if HN_KEY == BEST else '')] = [(BEST, HN_KEY == BEST)]
if BEST != 'e5s': variants[f'P2_{BEST}' + ('_hn' if HN_KEY == BEST else '') + '+e5s'] = [(BEST, HN_KEY == BEST), ('e5s', False)]
for name, encs in variants.items():
    tags = []
    for key, hn in encs:
        t, d = enc_ref(key, hn); pipe_encode(key, t, d); tags.append(t)
    Fv, Fr = build_pipeline_feats(name, tags[0], tags[1] if len(tags) > 1 else None)
    Fv, Fr = add_ranks(Fv), add_ranks(Fr); feats = feat_cols(Fr)
    models = lgb_fit(Fr, feats)
    m = eval_rank(Fv, lgb_pred(models, Fv, feats)); log_result('pipeline', name, m)
    PIPE[name] = dict(encs=encs, tags=tags, metric=m)
    del Fv, Fr; gc.collect()
PBEST = max(PIPE, key=lambda n: PIPE[n]['metric']['R@35w'])
if PIPE[PBEST]['metric']['R@35w'] < PIPE['P0_e5s']['metric']['R@35w'] + MIN_GAIN: PBEST = 'P0_e5s'
print('pipeline variant for further stages:', PBEST, PIPE[PBEST])
if CLEANUP:
    for n in PIPE:
        if n != PBEST:
            for sp in ('val', 'rk'):
                f = Path(f'data/{TAG}feats_{n}_{sp}.parquet')
                if f.exists(): f.unlink()
    print(f'свободно: {disk_free_gb():.1f} ГБ')

## 6. Cross-encoder и второй проход ранкера
1. **Первый проход без утечки:** out-of-fold скоры LightGBM на rk-запросах (4 фолда по запросам). Для val — среднее фолд-моделей.
2. **Обучение cross-encoder** на группах из T-лога: (запрос, выбранное объявление, 7 hard negatives из майнинга на объявлениях T). Функция потерь — листовой softmax. rk/val-запросы и их объявления в обучение не попадают.
3. **Скоринг** топ-`CE_TOPK` первого прохода для rk и val.
4. **Второй проход LightGBM** на тех же строках. Признаки: все признаки первого прохода, нормированные внутри запроса скоры первого прохода и cross-encoder (ранг, z-score, отставание от максимума). Сырые скоры не используются: так устойчивее к подмене CE_T → CE_F на бенчмарке.

In [ ]:
from sklearn.model_selection import GroupKFold
def stage1_oof(Fr, Fv, feats, folds=4):
    oof = np.zeros(len(Fr)); pv = np.zeros(len(Fv)); models_all = []
    for k, (tr_i, te_i) in enumerate(GroupKFold(folds).split(Fr, groups=Fr.qi)):
        ms = lgb_fit(Fr.iloc[tr_i], feats, seeds=1)
        oof[te_i] = lgb_pred(ms, Fr.iloc[te_i], feats); pv += lgb_pred(ms, Fv, feats) / folds; models_all += ms
    return oof, pv

def topk_rows(Fr, s1, k):
    Fr = Fr.assign(s1=s1); Fr['s1_rank'] = Fr.groupby('qi').s1.rank(ascending=False, method='first').astype(np.float32)
    Fr = Fr[Fr.s1_rank <= k].reset_index(drop=True); g = Fr.groupby('qi').s1
    # шкала скоров OOF-моделей и полной модели разная -> во второй проход идут только нормированные внутри запроса величины
    Fr['s1_z'] = ((Fr.s1 - g.transform('mean')) / (g.transform('std') + 1e-6)).astype(np.float32)
    Fr['s1_gap'] = (Fr.s1 - g.transform('max')).astype(np.float32)
    return Fr

def ce_groups(pairs, mined, corpus, max_groups=CE_MAX_GROUPS, seed=0):
    rng = np.random.default_rng(seed); idx = rng.permutation(len(pairs))[:max_groups]; G = []
    for i in idx:
        c = mined[i]; hard = c[:40]
        neg = list(rng.choice(hard, min(CE_NEG - 1, len(hard)), replace=False)) if len(hard) else []
        while len(neg) < CE_NEG: neg.append(int(rng.choice(corpus)))
        G.append((pairs.qt.iat[i], int(pairs.iid.iat[i]), [int(x) for x in neg]))
    return G

def train_ce(groups, out_dir, init=CE_MODEL):
    if prepare_out(out_dir): return out_dir
    tok = AutoTokenizer.from_pretrained(init); model = AutoModelForSequenceClassification.from_pretrained(init, num_labels=1).to(DEV).float()
    if CE_GC: model.gradient_checkpointing_enable()
    model.train(); n = 1 + CE_NEG; nb = max(1, int(len(groups) / CE_BS_GROUPS * CE_EPOCHS))
    opt = torch.optim.AdamW(model.parameters(), lr=CE_LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, s / max(1, 0.05 * nb)) * max(0.0, (nb - s) / max(1, 0.95 * nb)))
    rng = np.random.default_rng(0); step, t0, lsum = 0, time.time(), 0.0
    while step < nb:
        order = rng.permutation(len(groups))
        for b0 in range(0, len(order) - CE_BS_GROUPS + 1, CE_BS_GROUPS):
            gb = [groups[j] for j in order[b0:b0 + CE_BS_GROUPS]]
            qs = [g[0] for g in gb for _ in range(n)]; ds = [ITEXT[x] for g in gb for x in [g[1]] + g[2]]
            b = tok(qs, ds, truncation='only_second', max_length=CE_LEN, padding=True, return_tensors='pt').to(DEV)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=BF16):
                logits = model(**b).logits.view(-1, n).float()
            loss = F.cross_entropy(logits, torch.zeros(len(gb), dtype=torch.long, device=DEV))
            opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            step += 1; lsum += loss.item()
            if step % 100 == 0 or step == nb:
                print(f'  CE step {step}/{nb} loss {lsum / (100 if step % 100 == 0 else step % 100):.4f} {time.time() - t0:.0f}s', flush=True); lsum = 0.0
            if step >= nb: break
    save_bf16(model, tok, out_dir)
    del model; gc.collect()
    if DEV == 'cuda': torch.cuda.empty_cache()
    return out_dir

@torch.no_grad()
def ce_score(model_dir, qtexts, iids, bs=512):
    tok = AutoTokenizer.from_pretrained(model_dir); model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEV).float().eval()
    ds = [ITEXT[i] for i in iids]; order = np.argsort([len(q) + len(d) for q, d in zip(qtexts, ds)]); out = np.empty(len(ds), np.float32); t0 = time.time()
    for b0 in range(0, len(order), bs):
        ix = order[b0:b0 + bs]
        b = tok([qtexts[i] for i in ix], [ds[i] for i in ix], truncation='only_second', max_length=CE_LEN, padding=True, return_tensors='pt').to(DEV)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=BF16):
            out[ix] = model(**b).logits.view(-1).float().cpu().numpy()
    print(f'  CE scored {len(ds)} pairs in {time.time() - t0:.0f}s', flush=True)
    del model; gc.collect()
    if DEV == 'cuda': torch.cuda.empty_cache()
    return out

def ce_feats(Fr, ce):
    Fr = Fr.assign(ce=ce); g = Fr.groupby('qi').ce
    Fr['ce_z'] = ((Fr.ce - g.transform('mean')) / (g.transform('std') + 1e-6)).astype(np.float32)
    Fr['ce_rank'] = g.rank(ascending=False, method='min').astype(np.float32)
    Fr['ce_gap'] = (Fr.ce - g.transform('max')).astype(np.float32)
    return Fr

P = PIPE[PBEST]
Fv, Fr = build_pipeline_feats(PBEST, *tags_of(P)); Fv, Fr = add_ranks(Fv), add_ranks(Fr); FEATS1 = feat_cols(Fr)
oof, pv = stage1_oof(Fr, Fv, FEATS1)
log_result('stage1', f'{PBEST} (OOF-folds on val)', eval_rank(Fv, pv))
TR2, VA2 = topk_rows(Fr, oof, CE_TOPK), topk_rows(Fv, pv, CE_TOPK)
print('val recall inside stage-1 top-K:', round((VA2.groupby('qi').y.sum().reindex(range(len(VAL))).fillna(0).values / NREL_VAL).mean(), 4))

CE_T = train_ce(ce_groups(PAIRS_T, MINED_T, TCORP), f'models/{TAG}ce_T')
cev = f'data/{TAG}ce_T_{PBEST}_val.npy'; cer = f'data/{TAG}ce_T_{PBEST}_rk.npy'
if not Path(cev).exists(): np.save(cev, ce_score(CE_T, [VAL_QT[i] for i in VA2.qi], VCORP[VA2.ci.values]))
if not Path(cer).exists(): np.save(cer, ce_score(CE_T, [RK_QT[i] for i in TR2.qi], VCORP[TR2.ci.values]))
VA2, TR2 = ce_feats(VA2, np.load(cev)), ce_feats(TR2, np.load(cer))
log_result('ce-only', 'cross-encoder score on stage-1 top-K', eval_rank(VA2, VA2.ce.values))
FEATS2 = FEATS1 + ['s1_rank', 's1_z', 's1_gap', 'ce_z', 'ce_rank', 'ce_gap']
m2 = lgb_fit(TR2, FEATS2)
M2 = eval_rank(VA2, lgb_pred(m2, VA2, FEATS2)); log_result('stage2', f'{PBEST}+CE', M2)
M1 = RESULTS[[r['name'] for r in RESULTS].index(f'{PBEST} (OOF-folds on val)')]
USE_CE = M2['R@35w'] >= M1['R@35w'] + MIN_GAIN
if SMOKE: USE_CE = True   # в smoke-режиме прогоняем и ветку финала с cross-encoder
print('USE_CE =', USE_CE)

## 7. Сводка валидации

In [ ]:
res = pd.DataFrame(RESULTS); display(res) if 'display' in globals() else print(res.to_string())
print(f'\nИтоговая конфигурация: пайплайн {PBEST}, энкодеры (ключ, hard-neg) = {PIPE[PBEST]["encs"]}, cross-encoder = {USE_CE}')

## 8. Финал: бенчмарк
Выбранная конфигурация переобучается на всём train:
1. энкодер(ы) на полном логе (и второй раунд с hard negatives, если он помог);
2. эмбеддинги бенчмарка;
3. признаки: полный лог как история, корпус бенчмарка;
4. первый проход: ансамбль LightGBM на rk + val;
5. при `USE_CE`: cross-encoder (на всём train при `CE_FULL`) и второй проход, обученный на rk (OOF) + val;
6. правила категорий: 114 → только 114; cat-0 → до `R0` слотов под текстово-релевантные объявления вне 114.

Пишутся два файла: `subs/sub_server_stage1.csv` (запасной) и `subs/sub_server.csv` (основной, со вторым проходом, если он включён). Оба проверяются на формат.

In [ ]:
P = PIPE[PBEST]
if 'FEATS1' not in globals():   # быстрый путь: секция 6 (cross-encoder) пропущена
    _fr = pd.read_parquet(f'data/{TAG}feats_{PBEST}_rk.parquet'); FEATS1 = feat_cols(add_ranks(_fr)); del _fr
    USE_CE = False; print('секция 6 пропущена: финал без cross-encoder')
def full_encoder(key, hn):
    """энкодер на всём train (+ второй раунд hard-neg, если выбран) -> (tag, dir)"""
    d = train_biencoder(key, PAIRS_F, f'models/{TAG}{key}_F')
    if hn:
        mined = mine(key, d, PAIRS_F, LOG, FCORP, LP_F, cache=f'data/{TAG}mined_F_{key}.pkl')
        d = train_biencoder(key, PAIRS_F, f'models/{TAG}{key}_F_hn', init=d, hardneg=pick_hardneg(mined, FCORP))
        return f'{key}Fhn', d
    return f'{key}F', d
FTAGS, FDIRS = [], []
for key, hn in P['encs']:
    t, d = full_encoder(key, hn); pipe_encode(key, t, d, bench=True); FTAGS.append(t); FDIRS.append(d)
tf1, tf2 = FTAGS[0], (FTAGS[1] if len(FTAGS) > 1 else None)

fb = f'data/{TAG}feats_bench_{PBEST}.parquet'
if Path(fb).exists(): FB = pd.read_parquet(fb)
else:
    knn = (np.load('data/logQF_texts.npy', allow_pickle=True), np.load(emb_path(tf1, 'logQF')))
    B = Builder(BCORP, LOG, knn=knn)
    dense = {'d1': load_dense(tf1, 'bench')}
    if tf2: dense['d2'] = load_dense(tf2, 'bench')
    FB = par_build(B, BQ, dense, np.load(emb_path(tf1, 'benchQp')).astype(np.float32), labels=False)
    FB.to_parquet(fb, index=False); del B; gc.collect()
FB = add_ranks(FB)

# первый проход: rk + val
Fv, Fr = build_pipeline_feats(PBEST, *tags_of(P)); Fv, Fr = add_ranks(Fv), add_ranks(Fr)
Fall = pd.concat([Fr, Fv.assign(qi=Fv.qi + 10 ** 7)], ignore_index=True)
ms1 = lgb_fit(Fall, FEATS1); FB['s'] = lgb_pred(ms1, FB, FEATS1); del Fall; gc.collect()

def finalize(Fb, score_col, out_csv):
    Fb = Fb.copy(); qcat = BQ.search_category.values[Fb.qi.values]
    Fb = Fb[~((qcat == 114) & (Fb.i_cat114.values == 0))].sort_values(['qi', score_col], ascending=[True, False])
    qcat = BQ.search_category.values[Fb.qi.values]
    non = Fb[(qcat == 0) & (Fb.i_cat114.values == 0)]
    good = (non.bmn_all >= 0.4) | (non.rk_bm_all <= 20)
    for c in [c for c in Fb.columns if c.startswith('rk_dn_')]: good |= non[c] <= 20
    res_non = non[good].groupby('qi').head(R0)
    rest = Fb.drop(res_non.index)
    k_rest = (50 - res_non.groupby('qi').size()).reindex(range(len(BQ))).fillna(50).astype(int)
    rest = rest[rest.groupby('qi').cumcount() < k_rest.values[rest.qi.values]]
    top = pd.concat([res_non, rest]).sort_values(['qi', score_col], ascending=[True, False])
    ids = ITEMS.item_id.values
    ans = top.groupby('qi').ci.apply(lambda c: ' '.join(ids[BCORP[c.values]]))
    out = pd.DataFrame({'query_id': BQ.query_id.values, 'answer': ans.reindex(range(len(BQ))).fillna('').values})
    out.to_csv(out_csv, index=False); print('written', out_csv, out.shape, '| cat-0 queries with reserved non-114:', res_non.qi.nunique())
    if not SMOKE: run(f'{sys.executable} scripts/check_answer.py {out_csv}')
    else: assert (out.answer.str.split().str.len() == 50).all(), 'answer shorter than 50'

finalize(FB, 's', f'subs/{TAG}sub_server_stage1.csv')

if USE_CE:
    # второй проход: обучение на rk (OOF первого прохода) + val (скоры фолд-моделей), CE_T — для обучающих строк
    ce_b_dir = CE_T
    if CE_FULL:
        k1 = P['encs'][0][0]
        mined_F = mine(k1, FDIRS[0], PAIRS_F, LOG, FCORP, LP_F, cache=f'data/{TAG}mined_F_{FTAGS[0]}_ce.pkl')
        ce_b_dir = train_ce(ce_groups(PAIRS_F, mined_F, FCORP), f'models/{TAG}ce_F')
    TR2all = pd.concat([TR2, VA2.assign(qi=VA2.qi + 10 ** 7)], ignore_index=True)
    ms2 = lgb_fit(TR2all, FEATS2)
    FB2 = topk_rows(FB, FB.s.values, CE_TOPK)
    FB2 = ce_feats(FB2, ce_score(ce_b_dir, [BQ_QT[i] for i in FB2.qi], BCORP[FB2.ci.values]))
    FB2['s2'] = lgb_pred(ms2, FB2, FEATS2)
    # хвост за пределами top-K добираем из ранжирования первого прохода (всегда ниже любого s2), чтобы в ответе было ровно 50
    FBc = FB.merge(FB2[['qi', 'ci', 's2']], on=['qi', 'ci'], how='left')
    FBc['s2'] = FBc.s2.fillna(FBc.s - 1e6)
    finalize(FBc, 's2', f'subs/{TAG}sub_server.csv')
else:
    import shutil; shutil.copy(f'subs/{TAG}sub_server_stage1.csv', f'subs/{TAG}sub_server.csv'); print('CE not used: sub_server.csv = stage-1')

## 9. Итог
- **`subs/sub_server_stage1.csv`** — итоговый сабмит (отправлен как `answer.csv`, **Recall@50 = 0.9307**). Cross-encoder не прошёл порог по валидации, поэтому `subs/sub_server.csv` — его копия.
- `results/validation_metrics.csv` — все метрики прогона (тест энкодеров, варианты пайплайна, cross-encoder).

| Попытка | Конфигурация | `R@35w` на валидации | Лидерборд |
|---|---|---|---|
| 1 | e5-small, энкодер на всём train | 0.9261 | 0.9163 |
| 2 | Qwen + e5-small, энкодеры на половине train (быстрый финал, `experiments/06_notebook_addons/fast_final.py`) | 0.9390 | 0.9239 |
| 3 | **Qwen + e5-small, энкодеры на всём train (раздел 8)** | 0.9390 | **0.9307** |